# Employee Attrition Prediction - Model Training

This notebook trains three classification models: Logistic Regression, Decision Tree, and Random Forest.

## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import sys
sys.path.append('../src')
from model_training import ModelTrainer
import warnings
warnings.filterwarnings('ignore')

# Load preprocessed data
X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test = pd.read_csv('../data/y_test.csv').squeeze()

print(f"Train set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

## 2. Train Logistic Regression Model

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_model.fit(X_train, y_train)

# Get coefficients
coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print("Logistic Regression - Top 10 Features by Coefficient:")
print(coefficients.head(10))

# Save model
with open('../models/logistic_regression.pkl', 'wb') as f:
    pickle.dump(lr_model, f)
print("\nLogistic Regression model saved!")

## 3. Train Decision Tree Model

In [ ]:
# Train Decision Tree
dt_model = DecisionTreeClassifier(random_state=42, max_depth=10, min_samples_split=5, class_weight='balanced')
dt_model.fit(X_train, y_train)

# Get feature importance
dt_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Decision Tree - Top 10 Important Features:")
print(dt_importance.head(10))

# Save model
with open('../models/decision_tree.pkl', 'wb') as f:
    pickle.dump(dt_model, f)
print("\nDecision Tree model saved!")

## 4. Train Random Forest Model with Hyperparameter Tuning

In [ ]:
# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [10, 15, 20],
    'max_features': ['sqrt', 'log2']
}

# Grid search
rf_base = RandomForestClassifier(random_state=42, class_weight='balanced')
grid_search = GridSearchCV(rf_base, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best model
rf_model = grid_search.best_estimator_
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV F1 Score: {grid_search.best_score_:.4f}")

# Feature importance
rf_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nRandom Forest - Top 10 Important Features:")
print(rf_importance.head(10))

# Save model
with open('../models/random_forest.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
print("\nRandom Forest model saved!")

## 5. Compare Model Performance

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

models = {'Logistic Regression': lr_model, 'Decision Tree': dt_model, 'Random Forest': rf_model}
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba)
    })

comparison_df = pd.DataFrame(results)
print("Model Performance Comparison:")
print(comparison_df.to_string())

# Save comparison
comparison_df.to_csv('../results/model_comparison.csv', index=False)